In [1]:
import networkx as nx;
import pandas as pd;
import gurobipy as gp;
from gurobipy import GRB;
import csv;
import sys;
import numpy

In [2]:
networkCSV = 'TestInstances/CSV_TestInstances/N100/' + 'N100_22.csv';
N = 1000; #sample size
budget = 5;
numpy.random.seed(2024);

In [3]:
# Reading network file
with open(networkCSV, newline='') as f:
    reader = csv.reader(f);
    row1 = next(reader);
    nbArcs = int(row1[0]);
    row2 = next(reader);
    s = int(row2[0]);
    row3 = next(reader);
    t = int(row3[0]);
    
    G = nx.DiGraph();
    data = pd.read_csv(networkCSV, skiprows=4, header=None, delim_whitespace=True);
    n_edge = len(data.index);

    for i in range(n_edge): 
        G.add_edge(data.iat[i,0], data.iat[i,1], costLB = data.iat[i,2], 
                costUB = data.iat[i,3], interEffect = data.iat[i,4], tempCost = 0);

In [4]:
'''
print("nbArcs = ", nbArcs);
print("s = ", s);
print("t = ", t);
print("G.nodes = ", G.nodes)
print("G.edges = ", G.edges)
for e in G.edges:
    print(e)
    print(G.edges[e])
'''

'\nprint("nbArcs = ", nbArcs);\nprint("s = ", s);\nprint("t = ", t);\nprint("G.nodes = ", G.nodes)\nprint("G.edges = ", G.edges)\nfor e in G.edges:\n    print(e)\n    print(G.edges[e])\n'

In [5]:
scens = [];
for k in range(N):
    scen = {};
    for e in G.edges:
        scen[e] = numpy.random.uniform(G.edges[e]['costLB'],G.edges[e]['costUB']);
    scens.append(scen);

In [6]:
print("scens[0] = ", scens[0])

scens[0] =  {(1, 40): 386.0, (1, 61): 598.0, (1, 69): 680.0, (1, 82): 813.0, (1, 83): 817.0, (1, 93): 925.0, (1, 100): 995.0, (40, 2): 384.0, (40, 3): 370.4769140681644, (40, 5): 351.0, (40, 9): 309.0, (40, 25): 153.03118004650295, (40, 34): 57.048970780162534, (40, 48): 77.0, (40, 51): 109.0, (40, 55): 152.0, (40, 69): 293.3898157791806, (40, 74): 338.0, (40, 76): 365.0, (40, 79): 393.0, (40, 80): 401.0, (40, 89): 476.9108430772073, (40, 93): 534.0, (61, 37): 233.51767286768012, (61, 55): 53.95031167221101, (61, 77): 163.0, (61, 87): 267.9792339711498, (69, 10): 595.0, (69, 12): 566.0, (69, 13): 558.0, (69, 21): 480.4299286617262, (69, 26): 426.0, (69, 41): 279.0, (69, 52): 165.0, (69, 59): 100.0, (69, 61): 83.0, (69, 76): 65.0, (69, 94): 255.0, (82, 6): 765.0, (82, 9): 724.5153753519216, (82, 12): 700.0, (82, 33): 489.0, (82, 38): 440.0, (82, 55): 265.0, (82, 62): 198.0, (82, 89): 71.0, (82, 90): 79.03832374984464, (82, 92): 102.0, (83, 2): 811.0, (83, 7): 761.0, (83, 10): 733.0, (83

In [7]:
# Callback - use lazy constraints
def lazy(model, where):
    if where == GRB.Callback.MIPSOL:
        xvals = model.cbGetSolution(model._x)
        thetavals = model.cbGetSolution(model._theta);
        for k in range(len(model._scens)):
            # multi-cut version
            # update edge cost per scenario
            for e in model._G.edges:
                if xvals[e] > 1e-5:
                    model._G.edges[e]['tempCost'] = model._scens[k][e] + model._G.edges[e]['interEffect'];
                else:
                    model._G.edges[e]['tempCost'] = model._scens[k][e];
            # obtain the shortest path and its length
            spValue = nx.shortest_path_length(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
            if spValue < thetavals[k]-(1e-5):
                # add lazy constraints
                spPath = nx.shortest_path(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
                constrCoefList = [1];
                constrVarList = [model._theta[k]];
                rhs = 0
                for i in range(len(spPath)-1):
                    rhs += model._scens[k][(spPath[i],spPath[i+1])];
                    constrCoefList.append(-model._G.edges[(spPath[i],spPath[i+1])]['interEffect']);
                    constrVarList.append(model._x[(spPath[i],spPath[i+1])]);
                expr = gp.LinExpr();
                expr.addTerms(constrCoefList, constrVarList);
                model.cbLazy(expr <= rhs);

In [8]:
master = gp.Model()

# Create variables
x = {};
for e in G.edges:
    x[e] = master.addVar(obj=0, vtype=GRB.BINARY);
    
theta = {};
for k in range(N):
    theta[k] = master.addVar(obj=1.0/N, vtype=GRB.CONTINUOUS, lb = 0, ub = 1e7);

# Add interdiction budget constraint
master.addConstr(gp.quicksum(x[e] for e in G.edges) <= budget);

master._x = x
master._theta = theta
master._G = G
master._s = s
master._t = t
master._scens = scens

Set parameter Username
Academic license - for non-commercial use only - expires 2024-07-30


In [9]:
master.modelSense = GRB.MAXIMIZE
master.Params.LazyConstraints = 1
master.optimize(lazy)

xvals = master.getAttr('X', x)

print('')
print('Optimal objval: %g' % master.ObjVal)
print('')
print('Optimal xval = ')
for e in G.edges:
    if xvals[e] > 1e-5:
        print(e);
        print(" ")

Set parameter LazyConstraints to value 1
Gurobi Optimizer version 9.5.0 build v9.5.0rc5 (mac64[x86])
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 1 rows, 1999 columns and 999 nonzeros
Model fingerprint: 0x80bb8660
Variable types: 1000 continuous, 999 integer (999 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-03, 1e-03]
  Bounds range     [1e+00, 1e+07]
  RHS range        [5e+00, 5e+00]
Presolve time: 0.00s
Presolved: 1 rows, 1999 columns, 999 nonzeros
Variable types: 1000 continuous, 999 integer (999 binary)

Root simplex log...

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0187474e+03   1.979744e+03   0.000000e+00      9s
    1263    1.0057884e+03   0.000000e+00   0.000000e+00      9s

Root relaxation: objective 1.005788e+03, 1263 iterations, 0.03 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl